# Teste: Distributed Arithmetic (DA) na t1

No hls4ml 1.3.0, `Strategy = "distributed_arithmetic"` (biblioteca **da4ml**) troca as multiplicações
das camadas Dense/Conv por uma rede de **somas e deslocamentos**. Como os pesos são constantes
conhecidas na síntese, cada `x * w` vira somas de `x` deslocado (os bits 1 de `w`), e o da4ml
reaproveita subexpressões comuns entre todos os pesos da camada (ex.: `v36 = v0 + v9` é usado por
vários filtros). Resultado esperado:

- **0 DSP** e **sem ROM de pesos** (os pesos somem dentro da lógica): menos BRAM.
- Tudo em LUT/FF; o custo cresce com o nº de bits 1 dos pesos, então funciona melhor com
  **poucos bits** (t1 de 4 e 8 bits). No modelo sem quantização (`fixed<16,6>`) fica muito grande.
- **Só RF = 1** (totalmente paralelo): cada camada processa um pixel por ciclo, como a Latency com
  `rfmin`. Não dá para escolher RF/FPS alvo como nas outras estratégias; para comparar com o FINN
  use os pontos de maior FPS (alvo 50000).
- Bit-exato: a acurácia do hls4ml é a mesma do QKeras.

Este notebook sintetiza a t1 com DA em `t1_projects/` (mesmo nome que `python main.py --da` gera,
então o `relatorio.py` e o grupo `4_da` do `compara_finn.py` já as encontram) e compara com as
sínteses já feitas e com o FINN. Tabela e gráficos vão para `da_projects/`:

| configuração | de onde vem |
|---|---|
| Tudo em DA e convs em DA + Dense Resource (RF da Dense) | este notebook |
| Latency `rfmin` (dense Resource) | `t1_projects/`, a mais paralela das sínteses existentes |
| Latency `fps16271` (dense Resource) | `t1_projects/`, mesmo FPS alvo do FINN no alvo 50000 |
| FINN alvo 50000 | `dados_paper/fig6_recursos_finn.csv` |

In [ ]:
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import hls4ml
from qkeras.utils import load_qmodel

from cnn_base import prepare_input, set_input_precision
from relatorio import resumir_pasta
from main import apply_max_threads, save_accuracy
from utils import load_pickle_data, trace_layers
from vivado_utils import add_vivado_timing

## Configuração

In [ ]:
# (modelo em models/, Dense)
#   None              : a Dense também em DA (RF=1)
#   ("Resource", RF)  : convs em DA e a Dense 2048->6 em Resource com esse RF. A Dense leva ~256 + RF
#                       ciclos; com RF=4096 fica abaixo do II do resto da rede (~4624), sem limitar o FPS
# Na t1_8 a Dense em DA estoura a memória na otimização do da4ml (>27 GB em 17 min), por isso só a
# versão híbrida. Os modelos sem quantização (t1, t1_maxpool_bn) geram um DA ainda maior.
VARIANTES = [
    ("t1_4", None),
    ("t1_4", ("Resource", 4096)),
    ("t1_8", ("Resource", 4096)),
]

IO_TYPE = "io_stream"
CLOCK_PERIOD_NS = 10
FPGA_PART = "xc7z020clg400-1"
BACKEND = "Vitis"

PROJ_DIR = Path("t1_projects")   # mesmo lugar e nome que `main.py --da` usa
SAIDA = Path("da_projects")      # tabela e gráficos deste notebook
SAIDA.mkdir(exist_ok=True)
TEST_SIZE = 3000
MAX_THREADS = 1
RODAR_SINTESE = True          # False: só converte, compila e mede acurácia
PULAR_EXISTENTES = True       # não refaz variantes que já têm vivado_synth.rpt
BUILD = dict(reset=False, csim=False, synth=True, cosim=False, validation=False, export=False, vsynth=True)

# sínteses existentes usadas como referência
REFERENCIAS = {
    "Latency rfmin": "prj_{base}_io_stream_rfmin_Latency_denseResource",
    "Latency fps16271": "prj_{base}_io_stream_fps16271_Latency_denseResource",
}

## Conversão e síntese com DA

In [ ]:
def nome_projeto(base, dense):
    nome = f"prj_{base}_{IO_TYPE}_rf1_DA"
    if dense is not None:
        nome += f"_dense{dense[0]}{dense[1]}"
    return PROJ_DIR / nome


def rotulo_da(dense):
    # o DA só existe com RF=1; o RF do rótulo híbrido é o da Dense, que não está em DA
    return "Tudo em DA (RF=1)" if dense is None else f"Convs DA (RF=1) + Dense {dense[0]} (RF={dense[1]})"


def sintetizar_da(base, dense, X_test_raw, y_test):
    qmodel = load_qmodel(f"models/{base}/model.h5")
    X_test = np.ascontiguousarray(prepare_input(X_test_raw, qmodel))
    output_dir = nome_projeto(base, dense)
    output_dir.mkdir(parents=True, exist_ok=True)

    hls_config = hls4ml.utils.config_from_keras_model(
        qmodel, granularity="name", backend=BACKEND, default_reuse_factor=1,
    )
    hls_config["LayerName"]["output_dense"]["Precision"]["result"] = "fixed<16,6,RND,SAT>"
    set_input_precision(hls_config, qmodel)
    hls_config["Model"]["Strategy"] = "distributed_arithmetic"
    if dense is not None:
        hls_config["LayerName"]["output_dense"]["Strategy"] = dense[0]
        hls_config["LayerName"]["output_dense"]["ReuseFactor"] = dense[1]
    trace_layers(hls_config)

    hls_model = hls4ml.converters.convert_from_keras_model(
        qmodel,
        hls_config=hls_config,
        output_dir=str(output_dir),
        part=FPGA_PART,
        clock_period=CLOCK_PERIOD_NS,
        io_type=IO_TYPE,
        backend=BACKEND,
    )
    for layer in hls_model.get_layers():
        if layer.get_attr("strategy"):
            print(f"  {layer.name:14s} {layer.class_name:16s} strategy={layer.get_attr('strategy')}")

    hls_model.compile()
    save_accuracy(qmodel, hls_model, X_test, y_test, str(output_dir), TEST_SIZE)

    if RODAR_SINTESE:
        add_vivado_timing(str(output_dir))
        if MAX_THREADS is not None:
            apply_max_threads(str(output_dir), MAX_THREADS)
        hls_model.build(**BUILD)
    return output_dir

In [ ]:
X_test_raw, y_test = load_pickle_data("./dataset/sat6_test.pkl")

for base, dense in VARIANTES:
    print("\n" + "=" * 70 + f"\n{base}: {rotulo_da(dense)}\n" + "=" * 70)
    if PULAR_EXISTENTES and (nome_projeto(base, dense) / "vivado_synth.rpt").exists():
        print("já sintetizado, pulando")
        continue
    sintetizar_da(base, dense, X_test_raw, y_test)

## O que o da4ml gerou

O código de cada camada fica em `firmware/nnet_utils/nnet_code_gen.h` (uma função `dense_da_N` por
camada). Abaixo, quantas somas/subtrações o da4ml gerou contra quantas multiplicações a camada tem
por pixel (`n_in * n_out`).

In [ ]:
def operacoes_da(output_dir):
    txt = (Path(output_dir) / "firmware/nnet_utils/nnet_code_gen.h").read_text()
    linhas = []
    for m in re.finditer(r"void (dense_da_\d+)\(inp_t model_inp\[(\d+)\], out_t model_out\[(\d+)\]\)"
                         r" \{ // (.*?)\n(.*?)\n\}", txt, re.S):
        nome, n_in, n_out, tipos, corpo = m.groups()
        somas = len(re.findall(r"= [^;]*v\d+ [+-] ", corpo))
        linhas.append({"funcao": nome, "n_in": int(n_in), "n_out": int(n_out), "tipos": tipos,
                       "multiplicacoes": int(n_in) * int(n_out), "somas_da": somas})
    return pd.DataFrame(linhas)


for base, dense in VARIANTES:
    print(f"{base}: {rotulo_da(dense)}")
    display(operacoes_da(nome_projeto(base, dense)))

with open(nome_projeto(*VARIANTES[0]) / "firmware/nnet_utils/nnet_code_gen.h") as f:
    trecho = [l for l in f if re.search(r"= v\d+ [+-] v\d+", l)][:8]
print("Exemplo de código gerado:\n" + "".join(trecho))

## Comparação

In [ ]:
finn = pd.read_csv("dados_paper/fig6_recursos_finn.csv")
COLUNAS = ["modelo", "config", "fps", "lat_us", "lut", "ff", "bram", "dsp", "wns_ns", "acc", "cabe"]


def linha_hls(base, config, pasta):
    if not Path(pasta).is_dir():
        print(f"[AVISO] {pasta} não existe")
        return None
    _, r, _, _ = resumir_pasta(pasta)
    num = lambda k: pd.to_numeric(r.get(k), errors="coerce")
    periodo = max(CLOCK_PERIOD_NS, num("viv_periodo_ns") if pd.notna(num("viv_periodo_ns")) else 0)
    cabe = all(num(k) <= num(k + "_disp") for k in ("viv_LUT", "viv_FF", "viv_BRAM_tile", "viv_DSP")
               if pd.notna(num(k)))
    return {"modelo": base, "config": config, "fps": num("fps_vivado"),
            "lat_us": num("lat_max_cyc") * periodo / 1e3, "lut": num("viv_LUT"), "ff": num("viv_FF"),
            "bram": num("viv_BRAM_tile"), "dsp": num("viv_DSP"), "wns_ns": num("viv_wns_ns"),
            "acc": num("acc_segundo_valor"), "cabe": cabe}


MODELOS = list(dict.fromkeys(base for base, _ in VARIANTES))
linhas = []
for base in MODELOS:
    for b, dense in VARIANTES:
        if b == base:
            linhas.append(linha_hls(base, rotulo_da(dense), nome_projeto(base, dense)))
    for config, padrao in REFERENCIAS.items():
        linhas.append(linha_hls(base, config, Path("t1_projects") / padrao.format(base=base)))
    bits = re.match(r"t1_(\d+)$", base)
    if bits:
        f = finn[(finn["config"] == f"t1w{bits.group(1)}") & (finn["fps_alvo"] == 50000)].iloc[0]
        linhas.append({"modelo": base, "config": "FINN alvo 50000", "fps": f["fps_obtido"],
                       "lut": f["lut_finn"], "ff": f["ff_finn"], "bram": f["bram_finn"]})

comp = pd.DataFrame([l for l in linhas if l]).reindex(columns=COLUNAS)
comp.to_csv(SAIDA / "comparacao_da.csv", index=False)
comp.round({"fps": 0, "lat_us": 1, "lut": 0, "ff": 0, "bram": 1, "dsp": 0, "wns_ns": 2, "acc": 4})

In [ ]:
CORES = {"Tudo em DA (RF=1)": "#7a4fc9", "Latency rfmin": "#eb6834", "Latency fps16271": "#f5b293",
         "FINN alvo 50000": "#2a78d6"}
LIMITES = {"lut": 53200, "ff": 106400, "bram": 140}
PAINEIS = [("fps", "FPS (Vivado)"), ("lut", "LUT"), ("ff", "Flip-flops"), ("bram", "BRAM (tiles 36Kb)")]


def rotulo(v):
    return "" if pd.isna(v) else (f"{v / 1000:.1f}k" if v >= 1000 else f"{v:g}")


for base in MODELOS:
    d = comp[comp["modelo"] == base]
    fig, eixos = plt.subplots(1, len(PAINEIS), figsize=(16, 4))
    for ax, (col, nome) in zip(eixos, PAINEIS):
        x = np.arange(len(d))
        cores = [CORES.get(c, "#b59be3") for c in d["config"]]
        # vazado = não cabe no xc7z020
        faces = [c if ok is not False else "white" for c, ok in zip(cores, d["cabe"])]
        ax.bar(x, d[col].fillna(0), color=faces, edgecolor=cores, linewidth=1.5)
        for xi, v in zip(x, d[col]):
            ax.annotate(rotulo(v), (xi, 0 if pd.isna(v) else v), xytext=(0, 2),
                        textcoords="offset points", ha="center", fontsize=8)
        if col in LIMITES:
            ax.axhline(LIMITES[col], color="gray", linestyle="--", linewidth=1)
        ax.set_xticks(x)
        ax.set_xticklabels(d["config"].str.replace(" ", "\n", n=1).str.replace(" RF", "\nRF"), fontsize=8)
        ax.set_title(nome, loc="left", fontsize=10)
        ax.spines[["top", "right"]].set_visible(False)
        ax.margins(y=0.15)
    fig.suptitle(f"{base}: DA x Latency x FINN (linha tracejada = limite do xc7z020)", x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig(SAIDA / f"comparacao_da_{base}.png", dpi=150)
    plt.show()